# Accessing public data from Overture Maps Foundation and OpenStreetMap

In this notebook we will load data from publicly available sources using DuckDB and Python.

<a target="_blank" href="https://colab.research.google.com/github/kraina-ai/srai-tutorial/blob/dawts2026/01_access_public_data.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
## Uncomment the line below when running on Google Colab
# %pip install duckdb overturemaestro quackosm geopandas matplotlib

## Reading Overture Maps data using DuckDB

Here we will read the data directly from the S3 public bucket.

In [ ]:
import duckdb

duckdb.install_extension('spatial')
duckdb.load_extension('spatial')

In [ ]:
example_buildings_directory = "s3://overturemaps-us-west-2/release/2026-04-15.0/theme=buildings/type=building"
example_buildings_file = "part-00249-d40d7926-c563-5bd5-b457-4ccf7a6ee82e-c000.zstd.parquet"
example_buildings_full_path = f"{example_buildings_directory}/{example_buildings_file}"

In [ ]:
duckdb.sql(
    f"""
    SELECT *
    FROM '{example_buildings_full_path}'
    """
).show(max_width=250)

In [ ]:
duckdb.sql(
    f"""
    SELECT COUNT(*) total_buildings
    FROM '{example_buildings_full_path}'
    """
)

In [ ]:
duckdb.sql(
    f"""
    SELECT subtype, COUNT(*) total_buildings
    FROM '{example_buildings_full_path}'
    GROUP BY 1
    ORDER BY 2 DESC
    """
)

### Filter by bounding box

DuckDB will automatically scan whole directory of files and find files matching the bounding box filter.

In [ ]:
duckdb.sql(
    f"""
    SELECT *
    FROM '{example_buildings_full_path}'
    -- swap to full path below to scan a whole dataset, not just a single file
    -- FROM 's3://overturemaps-us-west-2/release/2026-04-15.0/theme=buildings/type=building/**/*.parquet'
    WHERE 
      bbox.xmin > 20.982 
      AND bbox.xmax < 21.028
      AND bbox.ymin > 52.218
      AND bbox.ymax < 52.246
    """
).show(max_width=250)

## Reading Overture Maps data using GeoPandas

This will load whole file into memory (original compressed file have almost 500mb).

Skip this cell if you don't have enough RAM for this operation.

In [ ]:
# import geopandas as gpd

# example_buildings_gdf = gpd.read_parquet(example_buildings_full_path)
# example_buildings_gdf

## Downloading Overture Maps data with OvertureMaestro

Here we will download buildings data only for the city of Warsaw using OvertureMaestro Python library.

In [ ]:
import overturemaestro as om

warsaw_buildings_omf = om.convert_geometry_to_geodataframe(
    theme="buildings",
    type="building",
    geometry_filter=om.geocode_to_geometry("Warsaw"),
)
warsaw_buildings_omf

In [ ]:
warsaw_buildings_omf.plot()

## Reading OpenStreetMap data using DuckDB - ST_ReadOSM

DuckDB has dedicated function for reading OSM data directly in raw format - `ST_ReadOSM`

In [ ]:
example_osm_full_path = 'https://download.bbbike.org/osm/bbbike/Warsaw/Warsaw.osm.pbf'

In [ ]:
duckdb.sql(
    f"""
    SELECT *
    FROM ST_ReadOSM('{example_osm_full_path}')
    WHERE tags['building'] IS NOT NULL 
    """
).show(max_width=250)

## Downloading OSM data with QuackOSM

Here we will download buildings data only for the city of Warsaw using QuackOSM Python library.

In [ ]:
import quackosm as qosm

warsaw_buildings_osm = qosm.convert_geometry_to_geodataframe(
    geometry_filter=qosm.geocode_to_geometry("Warsaw"),
    tags_filter={"building": True},  # filter is not required
)
warsaw_buildings_osm

In [ ]:
warsaw_buildings_osm.plot(markersize=0, linewidth=0)